# Lightstone validation report

Re-runs the analytical validation checks (docs/validation.md, Section 1) from Python and plots them. The authoritative pass/fail gates are the Rust tests (`cargo test`) and `tests/python`.

In [ ]:
import math, sys
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import jv, kv, jn_zeros
from scipy.optimize import brentq
import lightstone
sys.path.insert(0, '../reference')
import lightstone_ref

## V1 – step-index LP modes vs. Bessel eigenvalue equation (V = 4.5)

In [ ]:
lam, na, V = 850e-9, 0.1, 4.5
a = V * lam / (2 * math.pi * na)
res = lightstone.solve_modes({"core_radius": a, "na": na, "profile": {"kind": "step"}, "grid_points_per_radius": 800, "domain_factor": 8.0}, lam)
def b_exact(l, V):
    f = lambda u: u * jv(l - 1, u) / jv(l, u) + math.sqrt(V*V - u*u) * kv(l - 1, math.sqrt(V*V - u*u)) / kv(l, math.sqrt(V*V - u*u))
    z = [0.0] + list(jn_zeros(l, 10)); out = []
    for lo, hi in zip(z[:-1], z[1:]):
        if lo + 1e-9 < min(hi, V) - 1e-9:
            try: out.append(1 - (brentq(f, lo + 1e-9, min(hi, V) - 1e-9, xtol=1e-15) / V) ** 2)
            except ValueError: pass
    return out
n1, n2 = res["n_core"], res["n_clad"]
for m in res["modes"]:
    b = b_exact(m["l"], V)[m["m"] - 1]
    print(f"LP{m['l']}{m['m']}: Δn_eff = {abs(m['n_eff'] - math.sqrt(n2**2 + b*(n1**2 - n2**2))):.2e}")

## V2 – parabolic profile: mode-group spacing

For an infinite parabola, β² = k²n₁² − 2kn₁√(2Δ)·g/a.

In [ ]:
ms = lightstone.solve_modes({"profile": {"kind": "graded", "alpha": 2.0}, "grid_points_per_radius": 800})
k = 2 * math.pi / 850e-9; n1, n2 = ms["n_core"], ms["n_clad"]; d = (n1**2 - n2**2) / (2 * n1**2)
g = np.array([m["group"] for m in ms["modes"]]); ne = np.array([m["n_eff"] for m in ms["modes"]])
ex = np.sqrt(k*k*n1*n1 - 2*k*n1*math.sqrt(2*d)/25e-6*g) / k
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(g, ne, "o", label="solver"); ax[0].plot(g, ex, "-", label="infinite parabola"); ax[0].set(xlabel="group g", ylabel="n_eff"); ax[0].legend()
ax[1].semilogy(g, np.abs(ne - ex), "o"); ax[1].axhline(1e-5, color="r", ls="--"); ax[1].set(xlabel="group g", ylabel="|Δn_eff|", title="deviation grows near cutoff (finite core)")
plt.tight_layout()

## V4 – coaxial Gaussian on LP01 of a parabolic fiber

In [ ]:
w1 = math.sqrt(2 * (25e-6)**2 / ms["v_number"])
w0s = np.linspace(2e-6, 12e-6, 11); num = []
for w0 in w0s:
    c = lightstone.couple({"profile": {"kind": "graded", "alpha": 2.0}, "grid_points_per_radius": 400}, {"beam_radius": w0, "vcsel_mode_groups": 1})
    num.append(c["power"][0])
ana = (2 * w0s * w1 / (w0s**2 + w1**2))**2
print("max relative error:", np.max(np.abs(np.array(num) - ana) / ana))
plt.plot(w0s * 1e6, ana, "-", label="analytic"); plt.plot(w0s * 1e6, num, "o", label="Lightstone"); plt.xlabel("w0 [µm]"); plt.ylabel("LP01 coupling"); plt.legend();

## V8 – OM4 model bandwidths and DMD

Relative group delay per mode for the default OM4 preset (α = 2.0777, germania-doped core) and the OFL/EMB values.

In [ ]:
om4 = lightstone.solve_modes()
r = lightstone.simulate()
gd = np.array([m["group_delay"] for m in om4["modes"]]); gg = np.array([m["group"] for m in om4["modes"]])
plt.plot(gg, (gd - gd[0]) * 1e15, "o"); plt.xlabel("principal mode group"); plt.ylabel("relative delay [ps/km]")
print(f"OFL bandwidth: {om4['ofl_bandwidth_length']/1e9:.0f} MHz·km (OM4 min 3500)")
print(f"EMB (default launch): {r.metrics['emb_length']/1e9:.0f} MHz·km (OM4 min 4700)")

## V9/V10 – receiver noise calibration

In [ ]:
r0 = lightstone.simulate({"tx.p_avg_dbm": -40.0, "rx.rin_noise": False, "rx.shot_noise": False, "tx.pattern_periods": 8})
resid = r0.rx_signal - r0.rx_noiseless
print("thermal: measured/expected variance =", np.var(resid) / np.mean(r0.rx_sigma**2))
r1 = lightstone.simulate({"tx.pattern_periods": 8})
print("thermal+shot+RIN: measured/expected =", np.var(r1.rx_signal - r1.rx_noiseless) / np.mean(r1.rx_sigma**2))

## V15 – Rust vs NumPy reference

In [ ]:
cfg = lightstone.with_params(None, {"fiber.length": 120.0, "fiber.grid_points_per_radius": 300, "launch.offset": 4e-6, "rx.rin_noise": False, "rx.ffe_taps": 0})  # the reference has no FFE
rr, ref = lightstone.simulate(cfg), lightstone_ref.simulate_noiseless(cfg)
for k in ["tx_power", "rx_optical", "rx_noiseless", "reference_optical", "rx_sigma"]:
    a, b = getattr(rr, k), ref[k]
    print(f"{k:18s} max rel. diff {np.max(np.abs(a - b)) / np.max(np.abs(b)):.1e}")